# LURA — אימון מודל BERT

notebook עצמאי. אין תלות בקבצי הפרויקט — הכל כאן.

## לפני ההרצה

1. **Session options → Accelerator → GPU P100**
2. **Add Data → Upload** → העלי את שלושת הקבצים מ-`backend/ML/data/processed/`:
   `train.csv`, `val.csv`, `test.csv`
3. **Run All**

בסוף מורידים את `best_model.pt` ואת `best_model.meta.json` מלשונית Output,
ושמים אותם ב-`backend/ML/checkpoints/`.


In [ ]:
!pip install -q transformers


In [ ]:
import os, json, glob, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import f1_score, classification_report, roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if DEVICE.type != 'cuda':
    print('\n⚠ אין GPU. Session options → Accelerator → GPU P100, ואז Run All מחדש.')


## הגדרות

`MODEL_NAME` — להשוואה בין מודלים, שני מאותם 104 שפות:

| מודל | פרמטרים | מהירות |
|---|---|---|
| `bert-base-multilingual-cased` | 177M | בסיס |
| `distilbert-base-multilingual-cased` | 135M | פי ~2 |

על GPU האימון המלא לוקח 10–20 דקות, אז אין צורך לקצץ בנתונים.


In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
MAX_LENGTH = 256      # חייב להיות זהה בהרצה — נשמר במטא-דאטה
BATCH_SIZE = 32       # על P100 אפשר 32; אם נגמר הזיכרון, הורידי ל-16
EPOCHS     = 4
LR         = 2e-5
LIMIT      = 0        # 0 = כל סט האימון


## טעינת הנתונים

מאתר את הקבצים תחת `/kaggle/input` בלי תלות בשם ה-dataset.


In [ ]:
def find_csv(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(
            f'{name} לא נמצא. Add Data → Upload את train/val/test.csv')
    return hits[0]

splits = {}
for name in ['train', 'val', 'test']:
    df = pd.read_csv(find_csv(f'{name}.csv')).dropna(subset=['text', 'label'])
    splits[name] = (df['text'].astype(str).tolist(),
                    df['label'].astype(int).tolist())
    n = len(df); pos = int(df['label'].sum())
    print(f'{name:6} {n:>7} שורות  |  פישינג {pos} ({pos/n*100:.1f}%)')


In [ ]:
train_texts, train_labels = splits['train']
val_texts,   val_labels   = splits['val']
test_texts,  test_labels  = splits['test']

if LIMIT and LIMIT < len(train_texts):
    from sklearn.model_selection import train_test_split
    train_texts, _, train_labels, _ = train_test_split(
        train_texts, train_labels, train_size=LIMIT,
        stratify=train_labels, random_state=42)
    print(f'סט האימון צומצם ל-{len(train_texts)} שורות')

# כמה עברית יש בפועל — הבידול של הפרויקט
heb = sum(1 for t in train_texts if any('\u05d0' <= c <= '\u05ea' for c in t))
print(f'עברית בסט האימון: {heb} ({heb/len(train_texts)*100:.1f}%)')


## המודל


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2).to(DEVICE)

# DistilBERT ו-XLM-R אינם מקבלים token_type_ids
USES_TOKEN_TYPE = not any(k in MODEL_NAME.lower() for k in ('distil', 'xlm', 'minilm'))
print(f'{sum(p.numel() for p in model.parameters())/1e6:.0f}M פרמטרים  |  '
      f'token_type_ids: {USES_TOKEN_TYPE}')


In [ ]:
class EmailDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], max_length=MAX_LENGTH,
                        padding='max_length', truncation=True,
                        return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[i], dtype=torch.long)}
        if USES_TOKEN_TYPE and 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        return item

train_loader = DataLoader(EmailDataset(train_texts, train_labels),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(EmailDataset(val_texts, val_labels),
                          batch_size=BATCH_SIZE, num_workers=2)
test_loader  = DataLoader(EmailDataset(test_texts, test_labels),
                          batch_size=BATCH_SIZE, num_workers=2)
print(f'{len(train_loader)} batches ל-epoch')


## אימון

משקלי מחלקות מטפלים באי-האיזון (כ-40% פישינג), ולכן המודל לא לומד
לענות תמיד "לגיטימי". נשמר ה-checkpoint עם ה-F1 הגבוה ביותר על
סט הוולידציה, לא האחרון.


In [ ]:
counts = np.bincount(train_labels)
w = 1.0 / counts
class_weights = torch.tensor(w / w.sum() * len(counts),
                             dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

no_decay = ['bias', 'LayerNorm.weight']
optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
], lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * total_steps), total_steps)
print(f'{total_steps} צעדים בסך הכל')


In [ ]:
def run_eval(loader):
    model.eval()
    preds, labels, probs, loss_sum = [], [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            lbls = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            loss_sum += criterion(out.logits, lbls).item()
            p = torch.softmax(out.logits, dim=-1)
            preds.extend(out.logits.argmax(-1).cpu().numpy())
            probs.extend(p[:, 1].cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    return (loss_sum / len(loader),
            f1_score(labels, preds, pos_label=1, zero_division=0),
            float(np.mean(np.array(preds) == np.array(labels))),
            labels, preds, probs)


In [ ]:
best_f1 = 0.0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        lbls = batch.pop('labels').to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = criterion(out.logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f'  epoch {epoch}  step {step}/{len(train_loader)}  '
                  f'loss {running/step:.4f}  ({(time.time()-start)/60:.1f} דק\')',
                  flush=True)

    val_loss, val_f1, val_acc, *_ = run_eval(val_loader)
    print(f'epoch {epoch}  |  train {running/len(train_loader):.4f}  |  '
          f'val loss {val_loss:.4f}  F1 {val_f1:.4f}  acc {val_acc:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        with open('/kaggle/working/best_model.meta.json', 'w') as f:
            json.dump({'model_name': MODEL_NAME, 'max_length': MAX_LENGTH,
                       'epochs': EPOCHS, 'train_rows': len(train_texts),
                       'val_f1': round(val_f1, 4)}, f, indent=2)
        print(f'  ✓ נשמר (F1 {val_f1:.4f})')

print(f'\nסה"כ {(time.time()-start)/60:.1f} דקות')


## תוצאות על סט הבדיקה

המספרים כאן הם אלה שהולכים לדוח.


In [ ]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pt'))
loss, f1, acc, y, pred, prob = run_eval(test_loader)

tp = sum(1 for t, p in zip(y, pred) if t == 1 and p == 1)
fn = sum(1 for t, p in zip(y, pred) if t == 1 and p == 0)
fp = sum(1 for t, p in zip(y, pred) if t == 0 and p == 1)

print('=' * 56)
print(f'  דיוק      {acc*100:.2f}%')
print(f'  F1        {f1:.4f}')
print(f'  AUC-ROC   {roc_auc_score(y, prob):.4f}')
print(f'  פספוסים   {fn/max(tp+fn,1)*100:.2f}%   (FN={fn})')
print(f'  שווא      {fp/max(sum(1 for t in y if t==0),1)*100:.2f}%   (FP={fp})')
print('=' * 56)
print(classification_report(y, pred, target_names=['לגיטימי', 'פישינג'], digits=4))


## פילוח לפי שפה

המספר הכללי נשלט על ידי האנגלית. הפילוח הזה הוא מה שמראה אם
התמיכה בעברית עובדת באמת.


In [ ]:
is_heb = [any('\u05d0' <= c <= '\u05ea' for c in t) for t in test_texts]

for label, mask in [('עברית', is_heb), ('אנגלית', [not h for h in is_heb])]:
    yy = [t for t, m in zip(y, mask) if m]
    pp = [p for p, m in zip(pred, mask) if m]
    if not yy:
        print(f'{label}: אין דוגמאות'); continue
    a = float(np.mean(np.array(yy) == np.array(pp)))
    ff = f1_score(yy, pp, pos_label=1, zero_division=0)
    tp_ = sum(1 for t, p in zip(yy, pp) if t == 1 and p == 1)
    fn_ = sum(1 for t, p in zip(yy, pp) if t == 1 and p == 0)
    print(f'{label:8} {len(yy):>6} דוגמאות  |  דיוק {a*100:5.2f}%  |  '
          f'F1 {ff:.4f}  |  פספוסים {fn_/max(tp_+fn_,1)*100:5.2f}%')
    if len(yy) < 200:
        print(f'{"":8} ⚠ מדגם קטן — המספרים לא יציבים')


## הורדה

בלשונית **Output** בצד ימין יהיו:

- `best_model.pt`
- `best_model.meta.json`

**את שניהם** יש לשים ב-`backend/ML/checkpoints/`. קובץ המטא-דאטה אומר
לשרת באיזה מודל ובאיזה אורך רצף להשתמש — בלעדיו עלולה לחזור אי-ההתאמה
שכבר הייתה בפרויקט (אימון ב-256, הרצה ב-512).

ואז, מקומית:

```bash
cd backend
python ML/evaluate.py --split test
python ML/calibrate.py
```
